# Build trajactory using monocle 3 

In [ ]:
suppressPackageStartupMessages({
    library(Seurat)
    library(stringr)
    library(dplyr)
    library(patchwork)
    library(ggplot2)
    library(future)
    library(tidydr)
    library(harmony)
    library(reticulate)
    library(viridis)
    # library(speckle)
    library(limma)
    library(ggplot2)
    library(plyr)

    library(monocle3)
    library(SeuratWrappers)
    })


In [ ]:
## set working dir
dir = "/home/kibr/Work/Vascular_atlas"
setwd(dir)

In [ ]:
## load the data
h5ad_file="./Results/Revision_2/Endo_temp_processed.h5ad"
obj_oi = schard::h5ad2seurat(h5ad_file)
## check object
DefaultAssay(obj_oi) <- "RNA"
obj_oi
table(obj_oi$Cell_type)

In [ ]:
### The UMAP name should be correct for the object
DimPlot(obj_oi, group.by = c("individualID"),reduction = "umapharmony_")
DimPlot(obj_oi, group.by = c("Cell_type"),reduction = "umapharmony_")
# DimPlot(obj_oi, group.by = c("harmony_clusters"),reduction = "umapharmony_")

## Running monocle3 analysis

In [ ]:
# Preparing for the pseudobulk analysis
obj_oi <- JoinLayers(obj_oi)

In [ ]:
## Copy the "umapharmony_" embedding to "UMAP" enbedding in seurat object for the downstream analysis
obj_oi[["UMAP"]] <- obj_oi[["umapharmony_"]]


In [ ]:
## set up monocle3 cell_data_set object using the SeuratWrapper
# obj_oi <- JoinLayers(obj_oi)
cds <- as.cell_data_set(obj_oi)
# cds
cds <- cluster_cells(cds, resolution=1e-3,reduction_method = "UMAP")

In [ ]:
options(repr.plot.width = 15,repre.plot.height = 15)
p1 <- plot_cells(cds, color_cells_by = "cluster", show_trajectory_graph = FALSE,group_label_size = 10)
p2 <- plot_cells(cds, color_cells_by = "partition", show_trajectory_graph = FALSE)
p1
p2

### Trajectory analysis

In [ ]:
cds <- learn_graph(cds, use_partition = TRUE, verbose = FALSE)

In [ ]:
## After learning the graph, plot the trajectory graph to the cell plot
options(repr.plot.width = 15, repr.plot.height = 10)
plot_cells(cds,
           color_cells_by = "leiden_harmony",
           label_groups_by_cluster=FALSE,
           label_leaves=FALSE,
           label_branch_points=FALSE,
           group_label_size = 5)

plot_cells(cds,
           color_cells_by = "Cell_type",
           label_groups_by_cluster=FALSE,
           label_leaves=FALSE,
           label_branch_points=FALSE,
           group_label_size = 5)

### Colors cells by pseudotime

In [ ]:
## set the root to any one of the clusters by selecting the cells in that cluster to use as the root in the function order_cells. 
cds <- order_cells(cds, root_cells = colnames(cds[,clusters(cds) == 27]))

In [ ]:
## Plot
umap_theme <- theme_dr()+theme(panel.grid.major = element_blank(), 
                                            panel.grid.minor = element_blank(),
                                            panel.background = element_blank(), 
                                            axis.line = element_line(colour = "black"))

p <- plot_cells(cds,
           color_cells_by = "pseudotime",
           group_cells_by = "cluster",
           label_cell_groups = FALSE,
           label_groups_by_cluster=FALSE,
           label_leaves=FALSE,
           label_branch_points=FALSE,
           label_roots = FALSE,
           trajectory_graph_color = "grey60",
           show_trajectory_graph = F,raster = T) + umap_theme
p
# ggsave(filename = "./Results/subgroup/Mural_trajectory_umap.pdf",
#     patchwork::wrap_plots(p, ncol = 1),
#     scale = 1, width = 10, height = 6)

### Identify genes that change as a function of pseudotime

In [ ]:
cds <- estimate_size_factors(cds)
## Add gene names into CDS
cds@rowRanges@elementMetadata@listData[["gene_short_name"]] <- rownames(obj_oi[["RNA"]])

In [ ]:
cds_graph_test_results <- graph_test(cds,
                                     neighbor_graph = "principal_graph",
                                     cores = 8)

In [ ]:
write.csv(cds_graph_test_results,file ="./Results/Revision_2/DEG/Endo_ALL_trajectory_gene.csv")
# cds_graph_test_results <- read.csv("./Results/Revision_2/DEG/Endo_ALL_trajectory_gene.csv")

In [ ]:
check <- subset(cds_graph_test_results[order(cds_graph_test_results$morans_I, decreasing = TRUE),], q_value < 0.05)

## only keep those expressed
expr = LayerData(obj_oi, assay = "RNA", layer = "counts")
genes.percent.expression <- rowMeans(expr>0)
genes = names(genes.percent.expression[genes.percent.expression>0.05])
cat(paste(length(genes),"genes will be tested."))

check <- check[rownames(check) %in% genes,]
rownames(check[1:200,])

In [ ]:
rowData(cds)$gene_short_name <- row.names(rowData(cds))

# head(cds_graph_test_results, error=FALSE, message=FALSE, warning=FALSE)

deg_ids <- rownames(subset(cds_graph_test_results[order(cds_graph_test_results$morans_I, decreasing = TRUE),], q_value < 0.05))
head(deg_ids,n = 12)

plot_cells(cds,
           genes=head(deg_ids,n = 12),
           show_trajectory_graph = FALSE,
           label_cell_groups = FALSE,
           label_leaves = FALSE)

### Add the pseudotime value back to the seurat object

In [ ]:
### get the pseudotime point
time <- pseudotime(cds)

### assign value back to the seurat object
common_cells <- intersect(names(time),colnames(obj_oi))
obj_oi$pseudotime <- NA
obj_oi$pseudotime[common_cells] <- time[common_cells]

In [ ]:
p <- FeaturePlot(obj_oi, features = "pseudotime",reduction = "umapharmony_",,keep.scale = "all")
p
# ggsave(p,file = )

In [ ]:
saveRDS(obj_oi,file = "./Results/Revision_2/Endo_processed_pseudotime.rds")

In [ ]:
## modify the color of dots based on manually assigned colors for each cell type
col1=c(
  "Large_Artery"     = "#B2182B",  # deep red
  "Arterial"        = "#F4A582",  # light orange
  "Capillary"       = "#66C2A5",  # medium teal
  "Venous"           = "#4393C3",  # medium blue
  "Fenestrated_Capillary"   = "#FDAE61",  # golden orange (distinct capillary subtype)
  "EndoMT"          = "#BC80BD"  # soft lavender purple
)

p1 = DimPlot(obj_oi,group.by = c('Cell_type'),reduction = "umapharmony_",pt.size = 1.5,cols = col1,label = FALSE,label.size = 10)+umap_theme+NoLegend()
p2 = plot_cells(cds,
           color_cells_by = "pseudotime",
           cell_size = 1.2,
           label_cell_groups = FALSE,
           label_leaves = FALSE,
           label_branch_points = FALSE,
           label_roots = FALSE,
           trajectory_graph_color = "grey10",
           show_trajectory_graph = T,raster = T) + umap_theme

In [ ]:
options(repr.plot.width = 16,repr.plot.height = 6)
# p1+p2

ggsave(filename = "./Results/Revision_2/Figures/Figure_2X_Endothelial_cell_type_pseudotime_UMAP.pdf",
    patchwork::wrap_plots(p1,p2,ncol = 1),
      scale = 1, width = 12, height = 14)

<!-- ### Fit a quadratic linear regression for each expressed genes -->